In [1]:
import numpy as np
import bricks2marble as b2m

## Example Data

We simulate a model that returns a sequence of Tiberius-HMM states.
We set the total genome length to 3000 and divide this into three sequences of length 1000. A random fasta sequence of the same length will simulate the input to our model.

In [2]:
fasta = b2m.struct.FASTA([
    b2m.struct.Sequence(
        np.random.randint(9, size=(1000,)),
        name=f"seq_{i}",
        start=i*1000,
        end=(i+1)*1000,
    )
    for i in range(3)
]).resample(1000)

In [3]:
example_output = np.array([]
    # IR - exon - I0 - exon - I2 - exon - IR
    + 100*[0] + [7] + 20*[5, 6, 4] + [8] + 150*[1] + [11] + 10*[4, 5, 6]
    + [10] + 140*[3] + [13] + 30*[6, 4, 5] + [14] + 424*[0]

    # IR - exon - I0 - exon - I1 - exon - IR
    + 100*[0] + [7] + 15*[5, 6, 4] + [8] + 50*[1] + [11] + 40*[4, 5, 6]
    + [4, 5, 9] + 200*[2] + [12] + 35*[5, 6, 4] + [5, 14] + 500*[0]

    # IR - exon - I1 - exon - IR
    + 100*[0] + [7] + 15*[5, 6, 4] + [9] + 200*[2]
    + 35*[5, 6, 4] + [5, 14] + 417*[0]
)
example_output.shape

(3000,)

## Predict function
A pseudo predict function that would normally be a model call.

In [4]:
def predict_func(fasta: b2m.struct.FASTA) -> np.ndarray:
    returned = []
    for seq in fasta:
        for batch in seq.nuc:
            returned.append(example_output[seq.start:seq.end])
    return np.array(returned)

## Annotation of a fasta sequence
Using the above defined predict_func gives us a GTF annotation, which we can check by creating the
corresponding list of GTFEntry objects or writing it directly to a .gtf file.

In [5]:
annotation = b2m.GTF_from_model(
    fasta=fasta,
    predict_func=predict_func,
    model_name="Exampler",
    filter_transcripts=False,
)
# annotation.to_gtf("output.gtf")

In [6]:
entries = annotation.to_list()
entries

[GTFEntry(name='seq_0', start=101, end=576, strand='+', source='Exampler', feature=<FeatureType.Gene: 'gene'>, score=None, frame=None, attributes='gene_id "g1";'),
 GTFEntry(name='seq_0', start=101, end=576, strand='+', source='Exampler', feature=<FeatureType.Transcript: 'transcript'>, score=None, frame=None, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='seq_0', start=101, end=103, strand='+', source='Exampler', feature=<FeatureType.StartCodon: 'start_codon'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1";'),
 GTFEntry(name='seq_0', start=101, end=162, strand='+', source='Exampler', feature=<FeatureType.CDS: 'CDS'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1"; cds_type=initial;'),
 GTFEntry(name='seq_0', start=101, end=162, strand='+', source='Exampler', feature=<FeatureType.Exon: 'exon'>, score=None, frame=0, attributes='gene_id "g1"; transcript_id "g1.t1"; cds_type=initial;'),
 GTFEntry(name='seq_0', start=163, end